# CUDA Molecular Dynamics From Scratch: 1D Lennard-Jones on a T4

A molecular dynamics simulator for a one-dimensional chain of Lennard-Jones particles, written from scratch in C++ and CUDA. No library does any of the computation. The notebook starts with a two-particle CPU program that can be checked by hand and ends with a 65,536-particle GPU simulation that uses cell lists. Each step records what it did to correctness and to run time, including two bugs that turned up on the way.

This is part 1. Part 2 is a separate notebook on single versus double precision, a GPU-resident time loop and validation against OpenMM.

## Model

Reduced units throughout: sigma = epsilon = particle mass = 1.

$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right], \qquad F(r) = -\frac{dU}{dr} = \frac{24\varepsilon}{r}\left[2\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

F > 0 is repulsive. The force is zero at r = 2^(1/6) sigma, about 1.122.

Time stepping is velocity Verlet:

$$x(t+\Delta t) = x(t) + v(t)\Delta t + \tfrac12 a(t)\Delta t^2, \qquad v(t+\Delta t) = v(t) + \tfrac12\left[a(t) + a(t+\Delta t)\right]\Delta t$$

It is symplectic, so the total energy of a correct simulation oscillates around a constant instead of drifting. Total energy is the correctness check used throughout the notebook.

## Hardware and method

Everything runs in a Kaggle GPU session (NVIDIA T4). The T4 is an inference card with very little double-precision hardware: its FP64 peak is 254.4 GFLOP/s, which is 1/32 of its FP32 rate (NVIDIA T4 datasheet). All kernels in this part use `double`, and "fraction of peak" means that 254.4 GFLOP/s figure.

Each code cell writes a source file with `%%writefile`, then compiles and runs it. Kernel times use CUDA events, with one untimed warm-up launch followed by 20 timed runs; the minimum and the mean are reported. Nsight Compute is installed on Kaggle but blocked (section 2), so there are no hardware counters, only timings.

## Contents

1. CPU reference
2. First CUDA kernel
3. Fewer divisions, and a bug the spot-check caught
4. Shared-memory tiling
5. Cell lists
6. Multi-step dynamics and the energy drift
7. Summary

## 1. CPU reference

Two particles at x = 0 and x = 1.5, both at rest, one velocity Verlet step with dt = 0.01. This is small enough to check by hand.

In [1]:
%%writefile main.cpp
#include <iostream>
#include <cmath>

using namespace std;

double ljForce(double r, double sigma, double epsilon) {
    double x = sigma / r;
    double x6 = pow(x, 6);
    double x12 = x6 * x6;

    return (24.0 * epsilon / r) * (2.0 * x12 - x6);
}

int main() {
    double sigma = 1.0;
    double epsilon = 1.0;
    double m = 1.0;
    double dt = 0.01;

    double r1 = 0.0;
    double r2 = 1.5;

    double v1 = 0.0;
    double v2 = 0.0;

    double r = r2 - r1;

    double F = ljForce(r, sigma, epsilon);

    double direction1 = (r1 - r2) / fabs(r1 - r2);

    double F1 = F * direction1;
    double F2 = -F1;

    double a1 = F1 / m;
    double a2 = F2 / m;

    double r1_new = r1 + v1 * dt + 0.5 * a1 * dt * dt;
    double r2_new = r2 + v2 * dt + 0.5 * a2 * dt * dt;

    double r_new = r2_new - r1_new;

    double F_new = ljForce(r_new, sigma, epsilon);

    double direction1_new =
        (r1_new - r2_new) / fabs(r1_new - r2_new);

    double F1_new = F_new * direction1_new;
    double F2_new = -F1_new;

    double a1_new = F1_new / m;
    double a2_new = F2_new / m;

    double v1_new = v1 + 0.5 * (a1 + a1_new) * dt;
    double v2_new = v2 + 0.5 * (a2 + a2_new) * dt;

    cout << "After one velocity Verlet step:\n";
    cout << "r1 = " << r1_new << "\n";
    cout << "r2 = " << r2_new << "\n";
    cout << "v1 = " << v1_new << "\n";
    cout << "v2 = " << v2_new << "\n";

    return 0;
}

In [2]:
!g++ main.cpp -o main
!./main

After one velocity Verlet step:
r1 = 5.79014e-05
r2 = 1.49994
v1 = 0.0115828
v2 = -0.0115828


Three particles at x = 0, 1.2, 2.6 with initial velocities 0.1, -0.05, 0, for 1000 steps with dt = 0.001. Forces are summed over each pair once and applied with opposite signs (Newton's third law). Energy is printed every 100 steps.

In [3]:
%%writefile main4.cpp
#include <iostream>
#include <cmath>

using namespace std;

double ljPotential(double r, double sigma, double epsilon) {
    double x = sigma / r;
    double x6 = pow(x, 6);
    double x12 = x6 * x6;

    return 4.0 * epsilon * (x12 - x6);
}

double ljForce(double r, double sigma, double epsilon) {
    double x = sigma / r;
    double x6 = pow(x, 6);
    double x12 = x6 * x6;

    return (24.0 * epsilon / r) * (2.0 * x12 - x6);
}

void computeForces(double r[], int N, double F[],
                   double sigma, double epsilon) {

    for (int i = 0; i < N; i++) {
        F[i] = 0.0;
    }

    for (int i = 0; i < N; i++) {
        for (int j = i + 1; j < N; j++) {

            double rij = r[i] - r[j];
            double distance = fabs(rij);

            double force = ljForce(distance, sigma, epsilon);

            double direction = rij / distance;

            double fij = force * direction;

            F[i] += fij;
            F[j] -= fij;
        }
    }
}

double computeKineticEnergy(double v[], int N, double m) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE += 0.5 * m * v[i] * v[i];
    }

    return KE;
}

double computePotentialEnergy(double r[], int N,
                              double sigma, double epsilon) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {
        for (int j = i + 1; j < N; j++) {

            double rij = r[i] - r[j];
            double distance = fabs(rij);

            PE += ljPotential(distance, sigma, epsilon);
        }
    }

    return PE;
}

int main() {

    const int N = 3;

    double sigma = 1.0;
    double epsilon = 1.0;
    double m = 1.0;

    double dt = 0.001;
    int steps = 1000;

    double r[3] = {0.0, 1.2, 2.6};

    double v[3] = {0.1, -0.05, 0.0};

    double F[3];
    double F_new[3];

    double a[3];
    double a_new[3];

    computeForces(r, N, F, sigma, epsilon);

    for (int i = 0; i < N; i++) {
        a[i] = F[i] / m;
    }

    double initialKE = computeKineticEnergy(v, N, m);
    double initialPE = computePotentialEnergy(r, N, sigma, epsilon);
    double initialE = initialKE + initialPE;

    cout << "Initial energy = " << initialE << "\n\n";

    for (int step = 1; step <= steps; step++) {

        for (int i = 0; i < N; i++) {
            r[i] = r[i]
                 + v[i] * dt
                 + 0.5 * a[i] * dt * dt;
        }

        computeForces(r, N, F_new, sigma, epsilon);

        for (int i = 0; i < N; i++) {
            a_new[i] = F_new[i] / m;
        }

        for (int i = 0; i < N; i++) {
            v[i] = v[i]
                 + 0.5 * (a[i] + a_new[i]) * dt;
        }

        for (int i = 0; i < N; i++) {
            a[i] = a_new[i];
        }

        if (step % 100 == 0) {

            double KE = computeKineticEnergy(v, N, m);
            double PE = computePotentialEnergy(r, N, sigma, epsilon);
            double E = KE + PE;

            cout << "Step " << step << "\n";
            cout << "KE = " << KE << "\n";
            cout << "PE = " << PE << "\n";
            cout << "E  = " << E << "\n";
            cout << "E - E0 = " << E - initialE << "\n\n";
        }
    }

    return 0;
}

In [4]:
!g++ main4.cpp -o main
!./main

Initial energy = -1.35831

Step 100
KE = 0.0662373
PE = -1.42455
E  = -1.35831
E - E0 = -3.28987e-07

Step 200
KE = 0.151578
PE = -1.50989
E  = -1.35831
E - E0 = -9.62437e-07

Step 300
KE = 0.235438
PE = -1.59375
E  = -1.35831
E - E0 = 9.7805e-07

Step 400
KE = 0.52141
PE = -1.87972
E  = -1.35831
E - E0 = -1.69832e-06

Step 500
KE = 0.496151
PE = -1.85448
E  = -1.35833
E - E0 = -2.01501e-05

Step 600
KE = 0.458552
PE = -1.81689
E  = -1.35833
E - E0 = -2.46897e-05

Step 700
KE = 0.181893
PE = -1.54019
E  = -1.35829
E - E0 = 1.45712e-05

Step 800
KE = 0.601089
PE = -1.95941
E  = -1.35832
E - E0 = -8.17058e-06

Step 900
KE = 0.374664
PE = -1.73297
E  = -1.35831
E - E0 = -1.5324e-07

Step 1000
KE = 0.160959
PE = -1.51927
E  = -1.35831
E - E0 = 3.23117e-07


E - E0 stays below about 2.5e-5 and changes sign, so there is no drift. That is the behaviour the symplectic integrator should give, and this program is the CPU baseline for everything that follows.

## 2. First CUDA kernel

One thread per particle. Each thread sums the force from all other particles. Pairs are not shared between threads through Newton's third law, because that would need atomic writes; the price is twice the arithmetic of the CPU loop.

The first test uses the three-particle system again.

In [5]:
%%writefile forces.cu
#include <cstdio>
#include <cuda_runtime.h>

__device__ inline double ljForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(double *r, double *F, int N, double sigma, double epsilon) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = ljForce(dist, sigma, epsilon);
            Fi += force * (dx / dist);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 3;
    const double sigma = 1.0;
    const double epsilon = 1.0;

    double h_r[N] = {0.0, 1.2, 2.6};
    double h_F[N];

    double *d_r, *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(d_r, h_r, N * sizeof(double), cudaMemcpyHostToDevice);

    computeForcesKernel<<<1, 32>>>(d_r, d_F, N, sigma, epsilon);

    cudaMemcpy(h_F, d_F, N * sizeof(double), cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++) {
        printf("F[%d] = %.10f\n", i, h_F[i]);
    }

    cudaFree(d_r);
    cudaFree(d_F);

    return 0;
}

In [6]:
!nvcc forces.cu -o forces
!./forces

F[0] = 2.2413810681
F[1] = -0.5396964427
F[2] = -1.7016846254


The three forces add up to zero, as Newton's third law requires.

Next, 1024 particles at random positions, compared against a CPU loop that adds the terms in the same order.

In [7]:
%%writefile forces.cu
#include <cstdio>
#include <random>
#include <cmath>
#include <cuda_runtime.h>

double cpuLJForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__device__ inline double ljForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;
    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(double *r, double *F, int N, double sigma, double epsilon) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = ljForce(dist, sigma, epsilon);
            Fi += force * (dx / dist);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 1024;
    const double sigma = 1.0;
    const double epsilon = 1.0;

    double h_r[N];
    double h_F_cpu[N];
    double h_F_gpu[N];

    std::mt19937 gen(42);
    std::uniform_real_distribution<double> dist(0.5, 10.0);

    for (int i = 0; i < N; i++) {
        h_r[i] = dist(gen) + i * 1e-6;
    }

    for (int i = 0; i < N; i++) {
        h_F_cpu[i] = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = h_r[i] - h_r[j];
            double d = dx > 0.0 ? dx : -dx;

            double force = cpuLJForce(d, sigma, epsilon);
            h_F_cpu[i] += force * (dx / d);
        }
    }

    double *d_r, *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(d_r, h_r, N * sizeof(double), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
        d_r, d_F, N, sigma, epsilon
    );

    cudaMemcpy(h_F_gpu, d_F, N * sizeof(double), cudaMemcpyDeviceToHost);

    double maxError = 0.0;

    for (int i = 0; i < N; i++) {
        double error = fabs(h_F_cpu[i] - h_F_gpu[i]);
        if (error > maxError) {
            maxError = error;
        }
    }

    printf("N = %d\n", N);
    printf("Max absolute difference = %.15e\n", maxError);

    cudaFree(d_r);
    cudaFree(d_F);

    return 0;
}

In [8]:
!nvcc forces.cu -o forces
!./forces

N = 1024
Max absolute difference = 0.000000000000000e+00


The difference is exactly zero. This is expected: the CPU and GPU execute the same sequence of IEEE double-precision operations, so the results are bit-identical.

### Timing

Counting the operations in `ljForce` and the kernel body gives 16 floating-point operations per pair (a division counts as one). The naive kernel visits every ordered pair, so the total is N(N-1) x 16 FLOPs.

At N = 1024 the launch is 4 blocks of 256 threads. The T4 has 40 streaming multiprocessors, so at most 4 of them have work.

In [9]:
%%writefile forces.cu
#include <cstdio>
#include <cmath>
#include <random>
#include <iomanip>
#include <iostream>
#include <cuda_runtime.h>

double cpuLJForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__device__ inline double ljForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(
    double *r,
    double *F,
    int N,
    double sigma,
    double epsilon
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) {
                continue;
            }

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = ljForce(dist, sigma, epsilon);
            Fi += force * (dx / dist);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 1024;
    const double sigma = 1.0;
    const double epsilon = 1.0;
    const int threadsPerBlock = 256;
    const int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    const int numRuns = 20;

    double h_r[N];
    double h_F_cpu[N];
    double h_F_gpu[N];

    std::mt19937 gen(42);
    std::uniform_real_distribution<double> distribution(0.5, 10.0);

    for (int i = 0; i < N; i++) {
        h_r[i] = distribution(gen) + i * 1e-6;
    }

    for (int i = 0; i < N; i++) {
        h_F_cpu[i] = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) {
                continue;
            }

            double dx = h_r[i] - h_r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = cpuLJForce(dist, sigma, epsilon);
            h_F_cpu[i] += force * (dx / dist);
        }
    }

    double *d_r;
    double *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(
        d_r,
        h_r,
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
        d_r,
        d_F,
        N,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Warm-up launch error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
            d_r,
            d_F,
            N,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);

        cudaEventSynchronize(stop);

        cudaEventElapsedTime(&times[run], start, stop);
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel execution error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaMemcpy(
        h_F_gpu,
        d_F,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    double maxError = 0.0;

    for (int i = 0; i < N; i++) {
        double errorValue = std::fabs(h_F_cpu[i] - h_F_gpu[i]);

        if (errorValue > maxError) {
            maxError = errorValue;
        }
    }

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    std::cout << std::fixed << std::setprecision(6);

    std::cout << "N = " << N << '\n';
    std::cout << "Blocks = " << numBlocks << '\n';
    std::cout << "Threads/block = " << threadsPerBlock << '\n';
    std::cout << "Total threads = "
              << numBlocks * threadsPerBlock << '\n';

    std::cout << "\nKernel times (ms):\n";

    for (int run = 0; run < numRuns; run++) {
        std::cout << "Run " << (run + 1)
                  << ": " << times[run] << " ms\n";
    }

    std::cout << "\nAverage = "
              << averageTime << " ms\n";

    std::cout << "Minimum = "
              << minTime << " ms\n";

    std::cout << "\nMax absolute CPU-GPU difference = "
              << std::scientific
              << maxError << '\n';

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_r);
    cudaFree(d_F);

    return 0;
}

In [10]:
!nvcc forces.cu -o forces
!./forces

N = 1024
Blocks = 4
Threads/block = 256
Total threads = 1024

Kernel times (ms):
Run 1: 9.076736 ms
Run 2: 6.212000 ms
Run 3: 6.208416 ms
Run 4: 6.207744 ms
Run 5: 6.207776 ms
Run 6: 6.206016 ms
Run 7: 6.207424 ms
Run 8: 6.208768 ms
Run 9: 6.211744 ms
Run 10: 6.208512 ms
Run 11: 6.205664 ms
Run 12: 6.205824 ms
Run 13: 6.207712 ms
Run 14: 6.221824 ms
Run 15: 6.225888 ms
Run 16: 6.210176 ms
Run 17: 6.206944 ms
Run 18: 6.207520 ms
Run 19: 6.208096 ms
Run 20: 6.205376 ms

Average = 6.353009 ms
Minimum = 6.205376 ms

Max absolute CPU-GPU difference = 0.000000e+00


Minimum 6.205 ms. With the FLOP formula above that is about 2.7 GFLOP/s, 1.1% of FP64 peak.

At N = 65,536 there are 256 blocks, more than the number of multiprocessors.

In [11]:
%%writefile forces.cu
#include <cstdio>
#include <random>
#include <iomanip>
#include <iostream>
#include <cuda_runtime.h>

__device__ inline double ljForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(
    double *r,
    double *F,
    int N,
    double sigma,
    double epsilon
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) {
                continue;
            }

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = ljForce(dist, sigma, epsilon);
            Fi += force * (dx / dist);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 65536;
    const int threadsPerBlock = 256;
    const int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    const int numRuns = 20;

    const double sigma = 1.0;
    const double epsilon = 1.0;

    const double FLOPs = 68718428160.0;
    const double FP64_PEAK_GFLOPS = 254.4;

    double *h_r = new double[N];

    std::mt19937 gen(42);
    std::uniform_real_distribution<double> distribution(0.5, 10.0);

    for (int i = 0; i < N; i++) {
        h_r[i] = distribution(gen) + i * 1e-6;
    }

    double *d_r;
    double *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(
        d_r,
        h_r,
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
        d_r,
        d_F,
        N,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Warm-up launch error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
            d_r,
            d_F,
            N,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        cudaEventElapsedTime(&times[run], start, stop);
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel execution error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    double minSeconds = minTime / 1000.0;

    double achievedGFLOPS =
        (FLOPs / minSeconds) / 1.0e9;

    double peakFraction =
        (achievedGFLOPS / FP64_PEAK_GFLOPS) * 100.0;

    std::cout << std::fixed << std::setprecision(6);

    std::cout << "N = " << N << '\n';
    std::cout << "Blocks = " << numBlocks << '\n';
    std::cout << "Threads/block = " << threadsPerBlock << '\n';
    std::cout << "Total threads = "
              << numBlocks * threadsPerBlock << '\n';

    std::cout << "\nFLOPs = "
              << std::fixed << std::setprecision(0)
              << FLOPs << '\n';

    std::cout << "\nKernel times (ms):\n";

    for (int run = 0; run < numRuns; run++) {
        std::cout << "Run " << (run + 1)
                  << ": " << std::fixed
                  << std::setprecision(6)
                  << times[run] << " ms\n";
    }

    std::cout << "\nAverage = "
              << std::fixed << std::setprecision(6)
              << averageTime << " ms\n";

    std::cout << "Minimum = "
              << minTime << " ms\n";

    std::cout << "Achieved throughput = "
              << std::setprecision(6)
              << achievedGFLOPS << " GFLOP/s\n";

    std::cout << "Fraction of FP64 peak = "
              << peakFraction << "%\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_r);
    cudaFree(d_F);

    delete[] h_r;

    return 0;
}

In [12]:
!nvcc forces.cu -o forces
!./forces

N = 65536
Blocks = 256
Threads/block = 256
Total threads = 65536

FLOPs = 68718428160

Kernel times (ms):
Run 1: 1514.229492 ms
Run 2: 1514.175903 ms
Run 3: 1514.183105 ms
Run 4: 1514.279541 ms
Run 5: 1514.068970 ms
Run 6: 1516.998291 ms
Run 7: 1515.920410 ms
Run 8: 1514.129150 ms
Run 9: 1514.078857 ms
Run 10: 1514.240234 ms
Run 11: 1514.338745 ms
Run 12: 1514.273193 ms
Run 13: 1514.234009 ms
Run 14: 1514.303467 ms
Run 15: 1516.199951 ms
Run 16: 1516.931152 ms
Run 17: 1514.212891 ms
Run 18: 1514.137573 ms
Run 19: 1514.266602 ms
Run 20: 1514.130127 ms

Average = 1514.666504 ms
Minimum = 1514.068970 ms
Achieved throughput = 45.386590 GFLOP/s
Fraction of FP64 peak = 17.840641%


1514 ms at 45.4 GFLOP/s, 17.8% of peak. Throughput is about 17 times higher than at N = 1024 even though the work grew by a factor of about 4000, which fits the picture of idle multiprocessors at the small size. The two sizes are not a before/after comparison, only a diagnostic of how the kernel scales.

### Profiler

Nsight Compute is installed but cannot read performance counters on Kaggle.

In [13]:
!ncu --launch-count 1 ./forces

==PROF== Connected to process 457 (/kaggle/working/forces)
==ERROR== ERR_NVGPUCTRPERM - The user does not have permission to access NVIDIA GPU Performance Counters on the target device 0. For instructions on enabling permissions and to get more information see https://developer.nvidia.com/ERR_NVGPUCTRPERM
N = 65536
Blocks = 256
Threads/block = 256
Total threads = 65536

FLOPs = 68718428160

Kernel times (ms):
Run 1: 1514.098633 ms
Run 2: 1514.952393 ms
Run 3: 1514.630005 ms
Run 4: 1515.132935 ms
Run 5: 1514.903564 ms
Run 6: 1519.311035 ms
Run 7: 1514.905273 ms
Run 8: 1514.665039 ms
Run 9: 1514.979126 ms
Run 10: 1514.680298 ms
Run 11: 1514.534912 ms
Run 12: 1514.772461 ms
Run 13: 1514.694824 ms
Run 14: 1517.088013 ms
Run 15: 1516.545044 ms
Run 16: 1514.687622 ms
Run 17: 1514.735718 ms
Run 18: 1514.729736 ms
Run 19: 1514.642700 ms
Run 20: 1514.643311 ms

Average = 1515.166748 ms
Minimum = 1514.098633 ms
Achieved throughput = 45.385701 GFLOP/s
Fraction of FP64 peak = 17.840291%
==PROF== D

`ERR_NVGPUCTRPERM` is a restriction set by the host driver, so it cannot be changed from inside the notebook. For the rest of the project, claims about what limits a kernel come from predicting a timing, changing one thing and measuring, not from profiler data.

## 3. Fewer divisions, and a bug the spot-check caught

The kernel above does three double-precision divisions per pair (`sigma / r`, `24 * epsilon / r` and `dx / dist`). FP64 division is much slower than multiplication on this card. Computing `invR = 1.0 / dist` once and reusing it removes two of them.

A rough cost model that charges a division 10 to 15 times a multiplication predicts a speedup between 1.4x and 2.3x.

The cell also adds a correctness spot-check: 20 particles are recomputed on the CPU and compared with the GPU result.

In [14]:
%%writefile forces.cu
#include <cstdio>
#include <cmath>
#include <random>
#include <iomanip>
#include <iostream>
#include <cuda_runtime.h>

double cpuLJForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__device__ inline double ljForceOptimized(
    double r,
    double sigma,
    double epsilon
) {
    double invR = 1.0 / r;
    double sr = sigma * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return 24.0 * epsilon * invR * (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(
    double *r,
    double *F,
    int N,
    double sigma,
    double epsilon
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) {
                continue;
            }

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double invR = 1.0 / dist;

            double sr = sigma * invR;
            double sr6 = sr * sr * sr * sr * sr * sr;
            double sr12 = sr6 * sr6;

            double force =
                24.0 * epsilon * invR *
                (2.0 * sr12 - sr6);

            Fi += force * (dx * invR);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 65536;
    const int threadsPerBlock = 256;
    const int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    const int numRuns = 20;

    const double sigma = 1.0;
    const double epsilon = 1.0;

    double h_r[N];
    double h_F_gpu[N];

    std::mt19937 gen(42);
    std::uniform_real_distribution<double> distribution(0.5, 10.0);

    for (int i = 0; i < N; i++) {
        h_r[i] = distribution(gen) + i * 1e-6;
    }

    double *d_r;
    double *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(
        d_r,
        h_r,
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
        d_r,
        d_F,
        N,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Warm-up error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
            d_r,
            d_F,
            N,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        cudaEventElapsedTime(&times[run], start, stop);
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaMemcpy(
        h_F_gpu,
        d_F,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );
cudaMemcpy(
    h_F_gpu,
    d_F,
    N * sizeof(double),
    cudaMemcpyDeviceToHost
);

int testIndices[20] = {
    0, 100, 200, 300, 400,
    500, 600, 700, 800, 900,
    1000, 2000, 4000, 8000, 12000,
    16000, 20000, 30000, 40000, 50000
};

double maxError = 0.0;

for (int k = 0; k < 20; k++) {
    int i = testIndices[k];

    double cpuFi = 0.0;

    for (int j = 0; j < N; j++) {
        if (j == i) continue;

        double dx = h_r[i] - h_r[j];
        double dist = dx > 0.0 ? dx : -dx;

        double sr = sigma / dist;
        double sr6 = sr * sr * sr * sr * sr * sr;
        double sr12 = sr6 * sr6;

        double force =
            (24.0 * epsilon / dist) *
            (2.0 * sr12 - sr6);

        cpuFi += force * (dx / dist);
    }

    double error = std::fabs(cpuFi - h_F_gpu[i]);

    if (error > maxError) {
        maxError = error;
    }
}

std::cout << "Max absolute error (20-particle spot-check) = "
          << std::scientific << maxError << '\n';

    std::cout << std::fixed << std::setprecision(6);

    std::cout << "N = " << N << '\n';
    std::cout << "Blocks = " << numBlocks << '\n';
    std::cout << "Threads/block = " << threadsPerBlock << '\n';

    std::cout << "\nKernel times (ms):\n";

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        std::cout << "Run " << run + 1
                  << ": " << times[run] << " ms\n";

        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    std::cout << "\nAverage = "
              << averageTime << " ms\n";

    std::cout << "Minimum = "
              << minTime << " ms\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_r);
    cudaFree(d_F);

    return 0;
}

In [15]:
!nvcc forces.cu -o forces
!./forces

Max absolute error (20-particle spot-check) = 1.645505e+63
N = 65536
Blocks = 256
Threads/block = 256

Kernel times (ms):
Run 1: 887.009460 ms
Run 2: 886.990845 ms
Run 3: 887.028137 ms
Run 4: 887.084717 ms
Run 5: 887.076172 ms
Run 6: 886.943604 ms
Run 7: 887.007019 ms
Run 8: 887.099976 ms
Run 9: 887.031555 ms
Run 10: 891.481384 ms
Run 11: 887.099365 ms
Run 12: 886.953735 ms
Run 13: 886.910461 ms
Run 14: 886.977905 ms
Run 15: 886.916504 ms
Run 16: 886.913818 ms
Run 17: 887.038696 ms
Run 18: 886.903625 ms
Run 19: 887.003296 ms
Run 20: 887.017456 ms

Average = 887.224426 ms
Minimum = 886.903625 ms


The kernel got faster (1514 ms to 887 ms, 1.71x, inside the predicted range), but the spot-check fails with an error of 1.6e63. That is not rounding noise.

The next cell prints the nearest neighbour of each spot-checked particle.

In [16]:
%%writefile forces.cu
#include <cstdio>
#include <cmath>
#include <random>
#include <iomanip>
#include <iostream>
#include <cuda_runtime.h>

double cpuLJForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__device__ inline double ljForceOptimized(
    double r,
    double sigma,
    double epsilon
) {
    double invR = 1.0 / r;
    double sr = sigma * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return 24.0 * epsilon * invR *
           (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(
    double *r,
    double *F,
    int N,
    double sigma,
    double epsilon
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double invR = 1.0 / dist;

            double sr = sigma * invR;
            double sr6 = sr * sr * sr * sr * sr * sr;
            double sr12 = sr6 * sr6;

            double force =
                24.0 * epsilon * invR *
                (2.0 * sr12 - sr6);

            Fi += force * (dx * invR);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 65536;
    const int threadsPerBlock = 256;
    const int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    const int numRuns = 20;

    const double sigma = 1.0;
    const double epsilon = 1.0;

    double *h_r = new double[N];
    double *h_F_gpu = new double[N];

    std::mt19937 gen(42);
    std::uniform_real_distribution<double> distribution(0.5, 10.0);

    for (int i = 0; i < N; i++) {
        h_r[i] = distribution(gen) + i * 1e-6;
    }

    int testIndices[20] = {
        0, 100, 200, 300, 400,
        500, 600, 700, 800, 900,
        1000, 2000, 4000, 8000, 12000,
        16000, 20000, 30000, 40000, 50000
    };

    double globalMinDist = 1e300;
    int globalI = -1;
    int globalJ = -1;

    std::cout << std::scientific << std::setprecision(12);

    for (int k = 0; k < 20; k++) {
        int i = testIndices[k];

        double nearestDist = 1e300;
        int nearestJ = -1;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = h_r[i] - h_r[j];
            double dist = dx > 0.0 ? dx : -dx;

            if (dist < nearestDist) {
                nearestDist = dist;
                nearestJ = j;
            }

            if (dist < globalMinDist) {
                globalMinDist = dist;
                globalI = i;
                globalJ = j;
            }
        }

        std::cout
            << "Particle " << i
            << " nearest neighbor = " << nearestJ
            << ", distance = " << nearestDist
            << '\n';
    }

    std::cout << "\nSmallest distance among spot-checked particles = "
              << globalMinDist << '\n';

    std::cout << "Offending pair: "
              << globalI << " and " << globalJ << '\n';

    std::cout << "Positions: "
              << std::setprecision(17)
              << h_r[globalI] << " , "
              << h_r[globalJ] << "\n\n";

    double *d_r;
    double *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(
        d_r,
        h_r,
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
        d_r,
        d_F,
        N,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Warm-up launch error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
            d_r,
            d_F,
            N,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);

        cudaEventSynchronize(stop);

        cudaEventElapsedTime(&times[run], start, stop);
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel execution error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaMemcpy(
        h_F_gpu,
        d_F,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    double maxError = 0.0;

    for (int k = 0; k < 20; k++) {
        int i = testIndices[k];

        double cpuFi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = h_r[i] - h_r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = cpuLJForce(
                dist,
                sigma,
                epsilon
            );

            cpuFi += force * (dx / dist);
        }

        double errorValue =
            std::fabs(cpuFi - h_F_gpu[i]);

        if (errorValue > maxError) {
            maxError = errorValue;
        }
    }

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    const double FLOPs = 68718428160.0;
    const double FP64_PEAK_GFLOPS = 254.4;

    double minSeconds = minTime / 1000.0;

    double achievedGFLOPS =
        (FLOPs / minSeconds) / 1.0e9;

    double peakFraction =
        (achievedGFLOPS / FP64_PEAK_GFLOPS) * 100.0;

    std::cout << "\nN = " << N << '\n';
    std::cout << "Blocks = " << numBlocks << '\n';
    std::cout << "Threads/block = " << threadsPerBlock << '\n';

    std::cout << "\nKernel times (ms):\n";

    for (int run = 0; run < numRuns; run++) {
        std::cout
            << "Run " << run + 1
            << ": " << std::fixed
            << std::setprecision(6)
            << times[run]
            << " ms\n";
    }

    std::cout << "\nAverage = "
              << averageTime << " ms\n";

    std::cout << "Minimum = "
              << minTime << " ms\n";

    std::cout << "Max absolute error (20-particle spot-check) = "
              << std::scientific
              << std::setprecision(12)
              << maxError << '\n';

    std::cout << "Achieved throughput = "
              << std::fixed
              << std::setprecision(6)
              << achievedGFLOPS
              << " GFLOP/s\n";

    std::cout << "Fraction of FP64 peak = "
              << peakFraction
              << "%\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_r);
    cudaFree(d_F);

    delete[] h_r;
    delete[] h_F_gpu;

    return 0;
}

In [17]:
!nvcc forces.cu -o forces
!./forces

Particle 0 nearest neighbor = 54411, distance = 5.196799734186e-05
Particle 100 nearest neighbor = 17898, distance = 1.425146145095e-06
Particle 200 nearest neighbor = 12106, distance = 1.164324750391e-06
Particle 300 nearest neighbor = 17875, distance = 7.971650159710e-05
Particle 400 nearest neighbor = 40499, distance = 3.290356191776e-05
Particle 500 nearest neighbor = 13990, distance = 1.110134800730e-05
Particle 600 nearest neighbor = 58870, distance = 6.378573317045e-05
Particle 700 nearest neighbor = 47647, distance = 1.490431734652e-05
Particle 800 nearest neighbor = 54502, distance = 1.786193239894e-05
Particle 900 nearest neighbor = 26800, distance = 5.966857682260e-05
Particle 1000 nearest neighbor = 43696, distance = 1.606998154924e-05
Particle 2000 nearest neighbor = 32813, distance = 1.015611989077e-04
Particle 4000 nearest neighbor = 52030, distance = 2.218877191051e-05
Particle 8000 nearest neighbor = 52804, distance = 1.258672551145e-04
Particle 12000 nearest neighbor 

15 of the 20 particles have a neighbour closer than 1e-4, and particle 200 and particle 12106 are only 1.16e-6 apart. The force scales like r^-13, so at that distance it is of order 1e78, and a one-ULP difference between the CPU and GPU distance is amplified into an enormous error.

The cause is the test data. Positions were drawn uniformly at random in [0.5, 10] with no minimum separation, so with 65,536 particles in that interval near-collisions are certain. A configuration like that would also blow up any real simulation within one step.

The fix is a lattice with bounded jitter, `r_i = 1.5 i + U(-0.2, 0.2)`. Neighbouring jitter ranges cannot overlap, so the minimum separation is at least 1.1 by construction, just below the force minimum at 1.122.

In [18]:
%%writefile forces.cu
#include <cstdio>
#include <cmath>
#include <random>
#include <iomanip>
#include <iostream>
#include <cuda_runtime.h>

double cpuLJForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__global__ void computeForcesKernel(
    double *r,
    double *F,
    int N,
    double sigma,
    double epsilon
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = r[i] - r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double invR = 1.0 / dist;

            double sr = sigma * invR;
            double sr6 = sr * sr * sr * sr * sr * sr;
            double sr12 = sr6 * sr6;

            double force =
                24.0 * epsilon * invR *
                (2.0 * sr12 - sr6);

            Fi += force * (dx * invR);
        }

        F[i] = Fi;
    }
}

int main() {
    const int N = 65536;
    const int threadsPerBlock = 256;
    const int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    const int numRuns = 20;

    const double sigma = 1.0;
    const double epsilon = 1.0;

    const double FLOPs = 68718428160.0;
    const double FP64_PEAK_GFLOPS = 254.4;

    double *h_r = new double[N];
    double *h_F_gpu = new double[N];

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        h_r[i] = i * spacing + jitter(gen);
    }

    int testIndices[20] = {
        0, 100, 200, 300, 400,
        500, 600, 700, 800, 900,
        1000, 2000, 4000, 8000, 12000,
        16000, 20000, 30000, 40000, 50000
    };

    double globalMinDist = 1e300;
    int globalI = -1;
    int globalJ = -1;

    std::cout << std::scientific << std::setprecision(12);

    for (int k = 0; k < 20; k++) {
        int i = testIndices[k];

        double nearestDist = 1e300;
        int nearestJ = -1;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = h_r[i] - h_r[j];
            double dist = dx > 0.0 ? dx : -dx;

            if (dist < nearestDist) {
                nearestDist = dist;
                nearestJ = j;
            }

            if (dist < globalMinDist) {
                globalMinDist = dist;
                globalI = i;
                globalJ = j;
            }
        }

        std::cout
            << "Particle " << i
            << " nearest neighbor = " << nearestJ
            << ", distance = " << nearestDist
            << '\n';
    }

    std::cout << "\nSmallest distance among spot-checked particles = "
              << globalMinDist << '\n';

    std::cout << "Offending pair: "
              << globalI << " and " << globalJ << '\n';

    std::cout << "Positions: "
              << std::setprecision(17)
              << h_r[globalI] << " , "
              << h_r[globalJ] << "\n\n";

    double *d_r;
    double *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(
        d_r,
        h_r,
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
        d_r,
        d_F,
        N,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Warm-up launch error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        computeForcesKernel<<<numBlocks, threadsPerBlock>>>(
            d_r,
            d_F,
            N,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);

        cudaEventSynchronize(stop);

        cudaEventElapsedTime(&times[run], start, stop);
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel execution error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaMemcpy(
        h_F_gpu,
        d_F,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    double maxError = 0.0;

    for (int k = 0; k < 20; k++) {
        int i = testIndices[k];

        double cpuFi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = h_r[i] - h_r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = cpuLJForce(
                dist,
                sigma,
                epsilon
            );

            cpuFi += force * (dx / dist);
        }

        double errorValue =
            std::fabs(cpuFi - h_F_gpu[i]);

        if (errorValue > maxError) {
            maxError = errorValue;
        }
    }

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    double minSeconds = minTime / 1000.0;

    double achievedGFLOPS =
        (FLOPs / minSeconds) / 1.0e9;

    double peakFraction =
        (achievedGFLOPS / FP64_PEAK_GFLOPS) * 100.0;

    std::cout << "\nN = " << N << '\n';
    std::cout << "Blocks = " << numBlocks << '\n';
    std::cout << "Threads/block = " << threadsPerBlock << '\n';
    std::cout << "Total threads = "
              << numBlocks * threadsPerBlock << '\n';

    std::cout << "\nKernel times (ms):\n";

    for (int run = 0; run < numRuns; run++) {
        std::cout
            << "Run " << run + 1
            << ": " << std::fixed
            << std::setprecision(6)
            << times[run]
            << " ms\n";
    }

    std::cout << "\nAverage = "
              << averageTime << " ms\n";

    std::cout << "Minimum = "
              << minTime << " ms\n";

    std::cout << "Max absolute error (20-particle spot-check) = "
              << std::scientific
              << std::setprecision(12)
              << maxError << '\n';

    std::cout << "Achieved throughput = "
              << std::fixed
              << std::setprecision(6)
              << achievedGFLOPS
              << " GFLOP/s\n";

    std::cout << "Fraction of FP64 peak = "
              << peakFraction
              << "%\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_r);
    cudaFree(d_F);

    delete[] h_r;
    delete[] h_F_gpu;

    return 0;
}

In [19]:
!nvcc forces.cu -o forces
!./forces

Particle 0 nearest neighbor = 1, distance = 1.254756721442e+00
Particle 100 nearest neighbor = 101, distance = 1.331842650623e+00
Particle 200 nearest neighbor = 199, distance = 1.485263343422e+00
Particle 300 nearest neighbor = 299, distance = 1.240942784582e+00
Particle 400 nearest neighbor = 399, distance = 1.555797093580e+00
Particle 500 nearest neighbor = 499, distance = 1.331630042541e+00
Particle 600 nearest neighbor = 599, distance = 1.309316795797e+00
Particle 700 nearest neighbor = 701, distance = 1.410138952967e+00
Particle 800 nearest neighbor = 801, distance = 1.547951780394e+00
Particle 900 nearest neighbor = 901, distance = 1.209859516734e+00
Particle 1000 nearest neighbor = 999, distance = 1.326161816226e+00
Particle 2000 nearest neighbor = 2001, distance = 1.499956993757e+00
Particle 4000 nearest neighbor = 3999, distance = 1.496935135910e+00
Particle 8000 nearest neighbor = 7999, distance = 1.327223854661e+00
Particle 12000 nearest neighbor = 12001, distance = 1.41505

Nearest neighbours are now at normal distances, the spot-check error is 4.4e-16 (about 2 ULP), and the minimum time is 891 ms. Timing moved by 0.5% compared with the broken data, so the speedup from fewer divisions was not an artefact of the bad positions (the FLOP count does not depend on the data). The 1.7x speedup stands.

## 4. Shared-memory tiling

Each block loads a tile of 256 positions into shared memory, one element per thread, with `__syncthreads()` after the load and again before the next tile overwrites it. Global-memory reads drop by a factor of 256.

The expected gain is small. The 512 KB position array already fits in the T4's 4 MB L2 cache, so most redundant reads are cheap, and the arithmetic is unchanged.

In [20]:
%%writefile forces_tiled.cu
#include <cstdio>
#include <cmath>
#include <random>
#include <iomanip>
#include <iostream>
#include <cuda_runtime.h>

#define TILE_SIZE 256

double cpuLJForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) * (2.0 * sr12 - sr6);
}

__global__ void computeForcesTiledKernel(
    double *r,
    double *F,
    int N,
    double sigma,
    double epsilon
) {
    __shared__ double tile[TILE_SIZE];

    int i = blockIdx.x * blockDim.x + threadIdx.x;
    double Fi = 0.0;

    for (int tileStart = 0; tileStart < N; tileStart += TILE_SIZE) {

        int loadIndex = tileStart + threadIdx.x;

        if (loadIndex < N) {
            tile[threadIdx.x] = r[loadIndex];
        } else {
            tile[threadIdx.x] = 0.0;
        }

        __syncthreads();

        if (i < N) {
            for (int k = 0; k < TILE_SIZE; k++) {
                int j = tileStart + k;

                if (j < N && j != i) {
                    double dx = r[i] - tile[k];
                    double dist = dx > 0.0 ? dx : -dx;

                    double invR = 1.0 / dist;

                    double sr = sigma * invR;
                    double sr6 = sr * sr * sr * sr * sr * sr;
                    double sr12 = sr6 * sr6;

                    double force =
                        24.0 * epsilon * invR *
                        (2.0 * sr12 - sr6);

                    Fi += force * (dx * invR);
                }
            }
        }

        __syncthreads();
    }

    if (i < N) {
        F[i] = Fi;
    }
}

int main() {
    const int N = 65536;
    const int threadsPerBlock = TILE_SIZE;
    const int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    const int numRuns = 20;

    const double sigma = 1.0;
    const double epsilon = 1.0;

    const double FLOPs = 68718428160.0;
    const double FP64_PEAK_GFLOPS = 254.4;

    double *h_r = new double[N];
    double *h_F_gpu = new double[N];

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        h_r[i] = i * spacing + jitter(gen);
    }

    int testIndices[20] = {
        0, 100, 200, 300, 400,
        500, 600, 700, 800, 900,
        1000, 2000, 4000, 8000, 12000,
        16000, 20000, 30000, 40000, 50000
    };

    double *d_r;
    double *d_F;

    cudaMalloc(&d_r, N * sizeof(double));
    cudaMalloc(&d_F, N * sizeof(double));

    cudaMemcpy(
        d_r,
        h_r,
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    computeForcesTiledKernel<<<numBlocks, threadsPerBlock>>>(
        d_r,
        d_F,
        N,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Warm-up launch error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        computeForcesTiledKernel<<<numBlocks, threadsPerBlock>>>(
            d_r,
            d_F,
            N,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        cudaEventElapsedTime(&times[run], start, stop);
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr << "Kernel execution error: "
                  << cudaGetErrorString(error) << '\n';
        return 1;
    }

    cudaMemcpy(
        h_F_gpu,
        d_F,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    double maxError = 0.0;

    for (int k = 0; k < 20; k++) {
        int i = testIndices[k];

        double cpuFi = 0.0;

        for (int j = 0; j < N; j++) {
            if (j == i) continue;

            double dx = h_r[i] - h_r[j];
            double dist = dx > 0.0 ? dx : -dx;

            double force = cpuLJForce(
                dist,
                sigma,
                epsilon
            );

            cpuFi += force * (dx / dist);
        }

        double errorValue =
            std::fabs(cpuFi - h_F_gpu[i]);

        if (errorValue > maxError) {
            maxError = errorValue;
        }
    }

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    double minSeconds = minTime / 1000.0;

    double achievedGFLOPS =
        (FLOPs / minSeconds) / 1.0e9;

    double peakFraction =
        (achievedGFLOPS / FP64_PEAK_GFLOPS) * 100.0;

    std::cout << std::scientific << std::setprecision(12);

    std::cout << "Max absolute error = "
              << maxError << '\n';

    std::cout << "\n"
              << std::fixed << std::setprecision(6);

    std::cout << "N = " << N << '\n';
    std::cout << "Blocks = " << numBlocks << '\n';
    std::cout << "Threads/block = " << threadsPerBlock << '\n';

    std::cout << "\nKernel times (ms):\n";

    for (int run = 0; run < numRuns; run++) {
        std::cout << "Run " << run + 1
                  << ": " << times[run]
                  << " ms\n";
    }

    std::cout << "\nAverage = "
              << averageTime << " ms\n";

    std::cout << "Minimum = "
              << minTime << " ms\n";

    std::cout << "Achieved throughput = "
              << achievedGFLOPS
              << " GFLOP/s\n";

    std::cout << "Fraction of FP64 peak = "
              << peakFraction
              << "%\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_r);
    cudaFree(d_F);

    delete[] h_r;
    delete[] h_F_gpu;

    return 0;
}

In [21]:
!nvcc forces_tiled.cu -o forces_tiled
!./forces_tiled

Max absolute error = 4.440892098501e-16

N = 65536
Blocks = 256
Threads/block = 256

Kernel times (ms):
Run 1: 893.913574 ms
Run 2: 893.941467 ms
Run 3: 893.914795 ms
Run 4: 893.876221 ms
Run 5: 893.884399 ms
Run 6: 893.902100 ms
Run 7: 894.037964 ms
Run 8: 893.925354 ms
Run 9: 893.931396 ms
Run 10: 893.977905 ms
Run 11: 893.994995 ms
Run 12: 890.384033 ms
Run 13: 885.675110 ms
Run 14: 885.723022 ms
Run 15: 885.649658 ms
Run 16: 885.701538 ms
Run 17: 885.702026 ms
Run 18: 885.628845 ms
Run 19: 885.655334 ms
Run 20: 885.625244 ms

Average = 890.452148 ms
Minimum = 885.625244 ms
Achieved throughput = 77.593123 GFLOP/s
Fraction of FP64 peak = 30.500441%


Minimum time 885.6 ms against 891.2 ms before (0.6%). The first runs of the tiled kernel are slower (about 894 ms) and then settle, which looks like the GPU clock ramping up. Comparing the last 8 runs of each kernel, after the clocks have settled, gives 895.35 ms (standard deviation 0.03 ms) for the previous kernel and 885.67 ms (0.04 ms) for the tiled one: 1.1% faster. The noise is far smaller than the gap, so the improvement is real, and it is small, as predicted.

## 5. Cell lists

Almost all of the N(N-1) pair evaluations are between particles that are too far apart to matter. Using a cutoff of r_c = 2.5 sigma turns the problem into O(N).

The cutoff is a real approximation: U(2.5 sigma) = -0.0163 epsilon, 1.6% of the well depth, so the forces are no longer identical to the full sum. Results must be compared against a brute-force calculation that uses the same cutoff.

The line is divided into cells of width L = r_c. Two particles in cells at least two apart are farther than L >= r_c from each other, so every pair inside the cutoff is in the same cell or in adjacent cells. Checking three cells per particle is therefore exact, not a heuristic.

### CPU version

Particles are binned by cell, then compared with a brute-force loop that applies the same cutoff.

In [22]:
%%writefile cell_list_cpu.cpp
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>

double ljForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) *
           (2.0 * sr12 - sr6);
}

int main() {
    const int N = 65536;
    const double sigma = 1.0;
    const double epsilon = 1.0;
    const double rc = 2.5;
    const double Lcell = 2.5;

    std::vector<double> r(N);
    std::vector<double> F_brute(N, 0.0);
    std::vector<double> F_cell(N, 0.0);

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] = i * spacing + jitter(gen);
    }

    double r_min = r[0];
    double r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    int numCells =
        static_cast<int>(
            std::ceil((r_max - r_min) / Lcell)
        );

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<std::vector<int>> cells(numCells);

    for (int i = 0; i < N; i++) {
        int cellIndex =
            static_cast<int>(
                std::floor((r[i] - r_min) / Lcell)
            );

        if (cellIndex < 0) {
            cellIndex = 0;
        }

        if (cellIndex >= numCells) {
            cellIndex = numCells - 1;
        }

        cells[cellIndex].push_back(i);
    }

    long long bruteChecks = 0;
    long long cellCandidateChecks = 0;
    long long cellAcceptedInteractions = 0;

    for (int i = 0; i < N; i++) {
        double Fi = 0.0;

        for (int j = 0; j < N; j++) {
            if (i == j) {
                continue;
            }

            bruteChecks++;

            double dx = r[i] - r[j];
            double dist = std::fabs(dx);

            if (dist < rc) {
                double force = ljForce(
                    dist,
                    sigma,
                    epsilon
                );

                Fi += force * (dx / dist);
            }
        }

        F_brute[i] = Fi;
    }

    for (int i = 0; i < N; i++) {
        double Fi = 0.0;

        int currentCell =
            static_cast<int>(
                std::floor((r[i] - r_min) / Lcell)
            );

        if (currentCell < 0) {
            currentCell = 0;
        }

        if (currentCell >= numCells) {
            currentCell = numCells - 1;
        }

        for (int offset = -1; offset <= 1; offset++) {
            int neighborCell = currentCell + offset;

            if (neighborCell < 0 ||
                neighborCell >= numCells) {
                continue;
            }

            for (int j : cells[neighborCell]) {
                if (i == j) {
                    continue;
                }

                cellCandidateChecks++;

                double dx = r[i] - r[j];
                double dist = std::fabs(dx);

                if (dist < rc) {
                    cellAcceptedInteractions++;

                    double force = ljForce(
                        dist,
                        sigma,
                        epsilon
                    );

                    Fi += force * (dx / dist);
                }
            }
        }

        F_cell[i] = Fi;
    }

    double maxAbsError = 0.0;
    int worstIndex = -1;

    for (int i = 0; i < N; i++) {
        double error =
            std::fabs(F_brute[i] - F_cell[i]);

        if (error > maxAbsError) {
            maxAbsError = error;
            worstIndex = i;
        }
    }

    std::cout << std::scientific
              << std::setprecision(15);

    std::cout << "N = " << N << '\n';
    std::cout << "r_min = " << r_min << '\n';
    std::cout << "r_max = " << r_max << '\n';
    std::cout << "L_cell = " << Lcell << '\n';
    std::cout << "r_c = " << rc << '\n';
    std::cout << "numCells = " << numCells << '\n';

    std::cout << "\nBrute-force checks = "
              << bruteChecks << '\n';

    std::cout << "Cell-list candidate checks = "
              << cellCandidateChecks << '\n';

    std::cout << "Cell-list accepted interactions = "
              << cellAcceptedInteractions << '\n';

    std::cout << "\nMax absolute difference = "
              << maxAbsError << '\n';

    std::cout << "Worst particle index = "
              << worstIndex << '\n';

    double checkRatio =
        static_cast<double>(bruteChecks) /
        static_cast<double>(cellCandidateChecks);

    std::cout << "Candidate-check reduction = "
              << checkRatio << "x\n";

    return 0;
}

In [23]:
!g++ -O2 cell_list_cpu.cpp -o cell_list_cpu
!./cell_list_cpu

N = 65536
r_min = 1.186171937151384e-01
r_max = 9.830232120007831e+04
L_cell = 2.500000000000000e+00
r_c = 2.500000000000000e+00
numCells = 39321

Brute-force checks = 4294901760
Cell-list candidate checks = 262142
Cell-list accepted interactions = 131070

Max absolute difference = 0.000000000000000e+00
Worst particle index = -1
Candidate-check reduction = 1.638387499904632e+04x


The difference to brute force is exactly zero. Both visit accepted pairs in the same order (particles enter cells in increasing index order and cells are scanned left to right), so they perform the same sequence of additions. The 131,070 accepted interactions equal 2(N-1), which is what a chain with two neighbours per particle should give. The cell list needs 16,384 times fewer distance checks than brute force.

### GPU version

The GPU uses a compressed sparse row (CSR) layout: `sortedR` holds the positions reordered by cell, `sortedIndex` maps each sorted slot back to the original particle, and `cellStart` holds the offsets of each cell. The list is built on the host. The kernel finds a particle's cell on the device, loops over the three neighbouring cells, and the result is mapped back to the original order afterwards.

In [24]:
%%writefile cell_list_gpu.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

double ljForce(double r, double sigma, double epsilon) {
    double sr = sigma / r;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return (24.0 * epsilon / r) *
           (2.0 * sr12 - sr6);
}

__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    double *F_sorted,
    int N,
    int numCells,
    double r_min,
    double Lcell,
    double rc,
    double sigma,
    double epsilon
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k < N) {
        double myR = sortedR[k];

        int myCell =
            (int)floor((myR - r_min) / Lcell);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        double Fi = 0.0;

        for (int c = myCell - 1; c <= myCell + 1; c++) {
            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c];
                 m < cellStart[c + 1];
                 m++) {

                if (m == k) {
                    continue;
                }

                double dx = myR - sortedR[m];
                double dist = fabs(dx);

                if (dist < rc) {
                    double invR = 1.0 / dist;

                    double sr = sigma * invR;
                    double sr6 =
                        sr * sr * sr * sr * sr * sr;
                    double sr12 = sr6 * sr6;

                    double force =
                        24.0 * epsilon * invR *
                        (2.0 * sr12 - sr6);

                    Fi += force * (dx * invR);
                }
            }
        }

        F_sorted[k] = Fi;
    }
}

int main() {
    const int N = 65536;
    const double sigma = 1.0;
    const double epsilon = 1.0;
    const double rc = 2.5;
    const double Lcell = 2.5;

    const int threadsPerBlock = TILE_SIZE;
    const int numBlocks =
        (N + threadsPerBlock - 1) / threadsPerBlock;

    const int numRuns = 20;

    std::vector<double> r(N);
    std::vector<double> F_cpu(N, 0.0);
    std::vector<double> F_sorted(N, 0.0);
    std::vector<double> F_gpu(N, 0.0);

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] = i * spacing + jitter(gen);
    }

    double r_min = r[0];
    double r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    int numCells =
        (int)ceil((r_max - r_min) / Lcell);

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(numCells, 0);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor((r[i] - r_min) / Lcell);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    std::vector<int> cellStart(numCells + 1, 0);

    for (int c = 0; c < numCells; c++) {
        cellStart[c + 1] =
            cellStart[c] + cellCount[c];
    }

    std::vector<int> cursor = cellStart;

    std::vector<double> sortedR(N);
    std::vector<int> sortedIndex(N);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor((r[i] - r_min) / Lcell);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot = cursor[cell];

        sortedR[slot] = r[i];
        sortedIndex[slot] = i;

        cursor[cell]++;
    }

    for (int i = 0; i < N; i++) {
        double Fi = 0.0;

        int myCell =
            (int)floor((r[i] - r_min) / Lcell);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {
            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c];
                 m < cellStart[c + 1];
                 m++) {

                int j = sortedIndex[m];

                if (j == i) {
                    continue;
                }

                double dx = r[i] - r[j];
                double dist = fabs(dx);

                if (dist < rc) {
                    double force =
                        ljForce(dist, sigma, epsilon);

                    Fi += force * (dx / dist);
                }
            }
        }

        F_cpu[i] = Fi;
    }

    double *d_sortedR;
    int *d_cellStart;
    double *d_F_sorted;

    cudaMalloc(
        &d_sortedR,
        N * sizeof(double)
    );

    cudaMalloc(
        &d_cellStart,
        (numCells + 1) * sizeof(int)
    );

    cudaMalloc(
        &d_F_sorted,
        N * sizeof(double)
    );

    cudaMemcpy(
        d_sortedR,
        sortedR.data(),
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_cellStart,
        cellStart.data(),
        (numCells + 1) * sizeof(int),
        cudaMemcpyHostToDevice
    );

    cellListForceKernel<<<numBlocks, threadsPerBlock>>>(
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        N,
        numCells,
        r_min,
        Lcell,
        rc,
        sigma,
        epsilon
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Warm-up launch error: "
            << cudaGetErrorString(error)
            << '\n';
        return 1;
    }

    cudaMemcpy(
        F_sorted.data(),
        d_F_sorted,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    for (int k = 0; k < N; k++) {
        int originalIndex = sortedIndex[k];
        F_gpu[originalIndex] = F_sorted[k];
    }

    double maxAbsError = 0.0;
    int worstIndex = -1;

    for (int i = 0; i < N; i++) {
        double err =
            fabs(F_cpu[i] - F_gpu[i]);

        if (err > maxAbsError) {
            maxAbsError = err;
            worstIndex = i;
        }
    }

    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float times[numRuns];

    for (int run = 0; run < numRuns; run++) {
        cudaEventRecord(start);

        cellListForceKernel<<<numBlocks, threadsPerBlock>>>(
            d_sortedR,
            d_cellStart,
            d_F_sorted,
            N,
            numCells,
            r_min,
            Lcell,
            rc,
            sigma,
            epsilon
        );

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        cudaEventElapsedTime(
            &times[run],
            start,
            stop
        );
    }

    error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Kernel execution error: "
            << cudaGetErrorString(error)
            << '\n';
        return 1;
    }

    float sum = 0.0f;
    float minTime = times[0];

    for (int run = 0; run < numRuns; run++) {
        sum += times[run];

        if (times[run] < minTime) {
            minTime = times[run];
        }
    }

    float averageTime = sum / numRuns;

    std::cout
        << std::scientific
        << std::setprecision(15);

    std::cout << "N = "
              << N << '\n';

    std::cout << "r_min = "
              << r_min << '\n';

    std::cout << "r_max = "
              << r_max << '\n';

    std::cout << "numCells = "
              << numCells << '\n';

    std::cout << "Blocks = "
              << numBlocks << '\n';

    std::cout << "Threads/block = "
              << threadsPerBlock << '\n';

    std::cout << "\nMax absolute CPU-GPU error = "
              << maxAbsError << '\n';

    std::cout << "Worst particle index = "
              << worstIndex << '\n';

    std::cout << "\nKernel times (ms):\n";

    for (int run = 0; run < numRuns; run++) {
        std::cout
            << "Run " << run + 1
            << ": "
            << std::fixed
            << std::setprecision(6)
            << times[run]
            << " ms\n";
    }

    std::cout << "\nAverage = "
              << averageTime
              << " ms\n";

    std::cout << "Minimum = "
              << minTime
              << " ms\n";

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [25]:
!nvcc cell_list_gpu.cu -o cell_list_gpu
!./cell_list_gpu

N = 65536
r_min = 1.186171937151384e-01
r_max = 9.830232120007831e+04
numCells = 39321
Blocks = 256
Threads/block = 256

Max absolute CPU-GPU error = 1.332267629550188e-15
Worst particle index = 16598

Kernel times (ms):
Run 1: 0.188960 ms
Run 2: 0.181056 ms
Run 3: 0.180192 ms
Run 4: 0.180192 ms
Run 5: 0.179488 ms
Run 6: 0.178528 ms
Run 7: 0.180544 ms
Run 8: 0.179328 ms
Run 9: 0.178560 ms
Run 10: 0.180256 ms
Run 11: 0.180256 ms
Run 12: 0.182432 ms
Run 13: 0.180192 ms
Run 14: 0.180000 ms
Run 15: 0.179520 ms
Run 16: 0.178528 ms
Run 17: 0.179616 ms
Run 18: 0.180576 ms
Run 19: 0.179616 ms
Run 20: 0.178720 ms

Average = 0.180328 ms
Minimum = 0.178528 ms


The GPU result agrees with the CPU cell list to 1.3e-15 (about 6 ULP). It is not zero because the GPU may add the two accepted terms of a particle in a different order.

The minimum kernel time is 0.1785 ms for 65,536 particles:

| Kernel | Minimum time | Speedup over it |
|---|---|---|
| Naive, 3 divisions per pair | 1514.07 ms | |
| One division per pair | 891.20 ms | |
| Tiled | 885.63 ms | |
| Cell list | 0.1785 ms | 4,961x vs tiled, 4,992x vs one division |

Two cautions on reading this table. First, the cell-list kernel applies a cutoff, so it computes a different (truncated) force than the kernels above it; the comparison is time to solution, not the same computation made faster. Second, GFLOP/s and fraction of peak stop being useful metrics here, because the cell list performs about 16,000 times fewer operations; wall-clock time to solution is what counts.

The measured speedup (about 5,000x) is below the 16,384x reduction in distance checks. This notebook does not isolate the reason; part 2 finds that this kernel is limited by double-precision arithmetic, not by launch overhead.

## 6. Multi-step dynamics and the energy drift

The cell-list force kernel is now wired into a full velocity Verlet loop. The cell list is rebuilt on the host every step, since particles move. N = 65,536, lattice-plus-jitter start, initial velocities alternating +0.01 and -0.01.

### First run: hard cutoff

dt = 0.001, 1000 steps, forces and energy both cut off sharply at r_c.

In [26]:
%%writefile md_simulation.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;
const double DT = 0.001;

double ljForce(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return 24.0 * EPSILON * invR *
           (2.0 * sr12 - sr6);
}

double ljPotential(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return 4.0 * EPSILON *
           (sr12 - sr6);
}

__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    double *F_sorted,
    int N,
    int numCells,
    double r_min
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell =
        (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    double Fi = 0.0;

    for (int c = myCell - 1; c <= myCell + 1; c++) {
        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c];
             m < cellStart[c + 1];
             m++) {

            if (m == k) {
                continue;
            }

            double dx = myR - sortedR[m];
            double dist = fabs(dx);

            if (dist < RC) {
                double invR = 1.0 / dist;

                double sr = SIGMA * invR;
                double sr6 =
                    sr * sr * sr * sr * sr * sr;
                double sr12 = sr6 * sr6;

                double force =
                    24.0 * EPSILON * invR *
                    (2.0 * sr12 - sr6);

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    double& r_min,
    double& r_max,
    int& numCells
) {
    int N = r.size();

    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    numCells =
        (int)ceil((r_max - r_min) / LCELL);

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(numCells, 0);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(numCells + 1, 0);

    for (int c = 0; c < numCells; c++) {
        cellStart[c + 1] =
            cellStart[c] + cellCount[c];
    }

    std::vector<int> cursor = cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot = cursor[cell];

        sortedR[slot] = r[i];
        sortedIndex[slot] = i;

        cursor[cell]++;
    }
}

void computeCutoffEnergy(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    int numCells,
    double r_min,
    double& PE
) {
    int N = r.size();

    PE = 0.0;

    for (int i = 0; i < N; i++) {
        int myCell =
            (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {
            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c];
                 m < cellStart[c + 1];
                 m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dx = r[i] - r[j];
                double dist = fabs(dx);

                if (dist < RC) {
                    PE += ljPotential(dist);
                }
            }
        }
    }
}

double computeKineticEnergy(
    const std::vector<double>& v
) {
    double KE = 0.0;

    for (double vi : v) {
        KE += 0.5 * MASS * vi * vi;
    }

    return KE;
}

void computeForcesGPU(
    const std::vector<double>& sortedR,
    const std::vector<int>& cellStart,
    std::vector<double>& F_sorted,
    std::vector<double>& F,
    const std::vector<int>& sortedIndex,
    double r_min,
    int numCells,
    double* d_sortedR,
    int* d_cellStart,
    double* d_F_sorted
) {
    int N = sortedR.size();

    cudaMemcpy(
        d_sortedR,
        sortedR.data(),
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_cellStart,
        cellStart.data(),
        (numCells + 1) * sizeof(int),
        cudaMemcpyHostToDevice
    );

    int blocks =
        (N + TILE_SIZE - 1) / TILE_SIZE;

    cellListForceKernel<<<blocks, TILE_SIZE>>>(
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        N,
        numCells,
        r_min
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Kernel launch error: "
            << cudaGetErrorString(error)
            << '\n';
        exit(1);
    }

    cudaMemcpy(
        F_sorted.data(),
        d_F_sorted,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    for (int k = 0; k < N; k++) {
        int originalIndex = sortedIndex[k];
        F[originalIndex] = F_sorted[k];
    }
}

int main() {
    const int N = 65536;
    const int STEPS = 1000;

    std::vector<double> r(N);
    std::vector<double> v(N);
    std::vector<double> a(N);
    std::vector<double> a_new(N);
    std::vector<double> F(N);
    std::vector<double> F_sorted(N);

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] =
            i * spacing +
            jitter(gen);
    }

    for (int i = 0; i < N; i++) {
        if (i % 2 == 0) {
            v[i] = 0.01;
        } else {
            v[i] = -0.01;
        }
    }

    std::vector<double> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    double r_min;
    double r_max;
    int numCells;

    buildCellList(
        r,
        sortedR,
        sortedIndex,
        cellStart,
        r_min,
        r_max,
        numCells
    );

    double *d_sortedR;
    int *d_cellStart;
    double *d_F_sorted;

    cudaMalloc(
        &d_sortedR,
        N * sizeof(double)
    );

    cudaMalloc(
        &d_cellStart,
        (numCells + 1) * sizeof(int)
    );

    cudaMalloc(
        &d_F_sorted,
        N * sizeof(double)
    );

    computeForcesGPU(
        sortedR,
        cellStart,
        F_sorted,
        F,
        sortedIndex,
        r_min,
        numCells,
        d_sortedR,
        d_cellStart,
        d_F_sorted
    );

    for (int i = 0; i < N; i++) {
        a[i] = F[i] / MASS;
    }

    double KE0 =
        computeKineticEnergy(v);

    double PE0;

    computeCutoffEnergy(
        r,
        sortedIndex,
        cellStart,
        numCells,
        r_min,
        PE0
    );

    double E0 = KE0 + PE0;

    std::cout
        << std::scientific
        << std::setprecision(12);

    std::cout << "Initial conditions\n";
    std::cout << "N = " << N << '\n';
    std::cout << "dt = " << DT << '\n';
    std::cout << "rc = " << RC << '\n';
    std::cout << "Lcell = " << LCELL << '\n';
    std::cout << "Initial KE = " << KE0 << '\n';
    std::cout << "Initial PE = " << PE0 << '\n';
    std::cout << "Initial E  = " << E0 << '\n';

    std::cout << "\n";
    std::cout
        << std::setw(8) << "Step"
        << std::setw(20) << "KE"
        << std::setw(20) << "PE"
        << std::setw(20) << "Total E"
        << std::setw(20) << "E-E0"
        << '\n';

    std::cout
        << std::setw(8) << "0"
        << std::setw(20) << KE0
        << std::setw(20) << PE0
        << std::setw(20) << E0
        << std::setw(20) << 0.0
        << '\n';

    for (int step = 1; step <= STEPS; step++) {

        for (int i = 0; i < N; i++) {
            r[i] +=
                v[i] * DT +
                0.5 * a[i] * DT * DT;
        }

        buildCellList(
            r,
            sortedR,
            sortedIndex,
            cellStart,
            r_min,
            r_max,
            numCells
        );

        cudaFree(d_cellStart);

        cudaMalloc(
            &d_cellStart,
            (numCells + 1) * sizeof(int)
        );

        computeForcesGPU(
            sortedR,
            cellStart,
            F_sorted,
            F,
            sortedIndex,
            r_min,
            numCells,
            d_sortedR,
            d_cellStart,
            d_F_sorted
        );

        for (int i = 0; i < N; i++) {
            a_new[i] = F[i] / MASS;
        }

        for (int i = 0; i < N; i++) {
            v[i] +=
                0.5 *
                (a[i] + a_new[i]) *
                DT;
        }

        if (step % 100 == 0) {
            double KE =
                computeKineticEnergy(v);

            double PE;

            computeCutoffEnergy(
                r,
                sortedIndex,
                cellStart,
                numCells,
                r_min,
                PE
            );

            double E = KE + PE;
            double deltaE = E - E0;

            std::cout
                << std::setw(8)
                << step
                << std::setw(20)
                << KE
                << std::setw(20)
                << PE
                << std::setw(20)
                << E
                << std::setw(20)
                << deltaE
                << '\n';
        }

        a.swap(a_new);
    }

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [27]:
!nvcc -O2 md_simulation.cu -o md_simulation
!./md_simulation

Initial conditions
N = 65536
dt = 1.000000000000e-03
rc = 2.500000000000e+00
Lcell = 2.500000000000e+00
Initial KE = 3.276799999999e+00
Initial PE = -2.460319741960e+04
Initial E  = -2.459992061960e+04

    Step                  KE                  PE             Total E                E-E0
       0  3.276799999999e+00 -2.460319741960e+04 -2.459992061960e+04  0.000000000000e+00
     100  3.604025161465e+02 -2.496032284770e+04 -2.459992033156e+04  2.880415850086e-04
     200  1.453692884896e+03 -2.605361518407e+04 -2.459992229917e+04 -1.679572076682e-03
     300  2.956715946042e+03 -2.755890248518e+04 -2.460218653914e+04 -2.265919539783e+00
     400  3.917267887248e+03 -2.853752111631e+04 -2.462025322907e+04 -2.033260946686e+01
     500  5.190365487997e+03 -2.984023124536e+04 -2.464986575736e+04 -4.994513776017e+01
     600  6.708762693911e+03 -3.138610833086e+04 -2.467734563695e+04 -7.742501735433e+01
     700  7.104057193999e+03 -3.180437799372e+04 -2.470032079973e+04 -1.004001801265e

The total energy falls monotonically and reaches -173.6 at t = 1. A symplectic integrator should give bounded oscillation, so something is wrong.

### Is it the time step?

The same run with a 100x smaller time step (dt = 1e-5, 100,000 steps, same physical time). A time-stepping error scales like dt^2, so this should shrink the drift by a factor of about 10,000. The run also prints `MinGap`, the smallest distance between neighbours, to check for a close collision.

In [28]:
%%writefile md_dt1e5.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

const double DT = 0.00001;
const int N = 65536;
const int STEPS = 100000;
const int CHECKPOINT = 5000;

double ljForce(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;

    double sr6 =
        sr * sr * sr * sr * sr * sr;

    double sr12 = sr6 * sr6;

    return 24.0 * EPSILON * invR *
           (2.0 * sr12 - sr6);
}

double ljPotential(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;

    double sr6 =
        sr * sr * sr * sr * sr * sr;

    double sr12 = sr6 * sr6;

    return 4.0 * EPSILON *
           (sr12 - sr6);
}

__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    double *F_sorted,
    int N,
    int numCells,
    double r_min
) {
    int k =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell =
        (int)floor(
            (myR - r_min) / LCELL
        );

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    double Fi = 0.0;

    for (int c = myCell - 1;
         c <= myCell + 1;
         c++) {

        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c];
             m < cellStart[c + 1];
             m++) {

            if (m == k) {
                continue;
            }

            double dx =
                myR - sortedR[m];

            double dist = fabs(dx);

            if (dist < RC) {
                double invR = 1.0 / dist;

                double sr =
                    SIGMA * invR;

                double sr6 =
                    sr * sr * sr * sr * sr * sr;

                double sr12 =
                    sr6 * sr6;

                double force =
                    24.0 * EPSILON * invR *
                    (2.0 * sr12 - sr6);

                Fi +=
                    force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    double& r_min,
    double& r_max,
    int& numCells
) {
    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min =
            std::min(r_min, r[i]);

        r_max =
            std::max(r_max, r[i]);
    }

    numCells =
        (int)ceil(
            (r_max - r_min) / LCELL
        );

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(
        numCells,
        0
    );

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor(
                (r[i] - r_min) / LCELL
            );

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(
        numCells + 1,
        0
    );

    for (int c = 0;
         c < numCells;
         c++) {

        cellStart[c + 1] =
            cellStart[c] +
            cellCount[c];
    }

    std::vector<int> cursor =
        cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor(
                (r[i] - r_min) / LCELL
            );

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot =
            cursor[cell];

        sortedR[slot] =
            r[i];

        sortedIndex[slot] =
            i;

        cursor[cell]++;
    }
}

void computeForcesGPU(
    const std::vector<double>& sortedR,
    const std::vector<int>& cellStart,
    const std::vector<int>& sortedIndex,
    std::vector<double>& F,
    double r_min,
    int numCells,
    double* d_sortedR,
    int* d_cellStart,
    double* d_F_sorted
) {
    cudaMemcpy(
        d_sortedR,
        sortedR.data(),
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_cellStart,
        cellStart.data(),
        (numCells + 1) * sizeof(int),
        cudaMemcpyHostToDevice
    );

    int blocks =
        (N + TILE_SIZE - 1) /
        TILE_SIZE;

    cellListForceKernel<<<
        blocks,
        TILE_SIZE
    >>>(
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        N,
        numCells,
        r_min
    );

    cudaError_t error =
        cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Kernel launch error: "
            << cudaGetErrorString(error)
            << '\n';

        exit(1);
    }

    std::vector<double> F_sorted(N);

    cudaMemcpy(
        F_sorted.data(),
        d_F_sorted,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    for (int k = 0; k < N; k++) {
        int originalIndex =
            sortedIndex[k];

        F[originalIndex] =
            F_sorted[k];
    }
}

double computeKE(
    const std::vector<double>& v
) {
    double KE = 0.0;

    for (int i = 0; i < N; i++) {
        KE +=
            0.5 *
            MASS *
            v[i] *
            v[i];
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells
) {
    double PE = 0.0;

    for (int i = 0; i < N; i++) {

        int myCell =
            (int)floor(
                (r[i] - r_min) /
                LCELL
            );

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1;
             c <= myCell + 1;
             c++) {

            if (c < 0 ||
                c >= numCells) {
                continue;
            }

            for (int m = cellStart[c];
                 m < cellStart[c + 1];
                 m++) {

                int j =
                    sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dx =
                    r[i] - r[j];

                double dist =
                    fabs(dx);

                if (dist < RC) {
                    PE += ljPotential(dist) - ljPotential(RC);
                }
            }
        }
    }

    return PE;
}

double minimumGap(
    const std::vector<double>& sortedR
) {
    double minDist =
        1e300;

    for (int i = 1; i < N; i++) {
        double gap =
            sortedR[i] -
            sortedR[i - 1];

        if (gap < minDist) {
            minDist = gap;
        }
    }

    return minDist;
}

int main() {

    std::vector<double> r(N);
    std::vector<double> v(N);
    std::vector<double> a(N);
    std::vector<double> a_new(N);
    std::vector<double> F(N);

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] =
            i * spacing +
            jitter(gen);
    }

    for (int i = 0; i < N; i++) {
        if (i % 2 == 0) {
            v[i] = 0.01;
        } else {
            v[i] = -0.01;
        }
    }

    std::vector<double> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    double r_min;
    double r_max;
    int numCells;

    buildCellList(
        r,
        sortedR,
        sortedIndex,
        cellStart,
        r_min,
        r_max,
        numCells
    );

    double* d_sortedR;
    int* d_cellStart;
    double* d_F_sorted;

    cudaMalloc(
        &d_sortedR,
        N * sizeof(double)
    );

    cudaMalloc(
        &d_cellStart,
        (numCells + 1) *
        sizeof(int)
    );

    cudaMalloc(
        &d_F_sorted,
        N * sizeof(double)
    );

    computeForcesGPU(
        sortedR,
        cellStart,
        sortedIndex,
        F,
        r_min,
        numCells,
        d_sortedR,
        d_cellStart,
        d_F_sorted
    );

    for (int i = 0; i < N; i++) {
        a[i] =
            F[i] / MASS;
    }

    double KE0 =
        computeKE(v);

    double PE0 =
        computePE(
            r,
            sortedIndex,
            cellStart,
            r_min,
            numCells
        );

    double E0 =
        KE0 + PE0;

    std::cout
        << std::scientific
        << std::setprecision(12);

    std::cout
        << "N = "
        << N
        << '\n';

    std::cout
        << "dt = "
        << DT
        << '\n';

    std::cout
        << "Total simulated time = "
        << DT * STEPS
        << '\n';

    std::cout
        << "Initial KE = "
        << KE0
        << '\n';

    std::cout
        << "Initial PE = "
        << PE0
        << '\n';

    std::cout
        << "Initial E  = "
        << E0
        << '\n';

    std::cout << '\n';

    std::cout
        << std::setw(10)
        << "Step"
        << std::setw(20)
        << "Time"
        << std::setw(20)
        << "MinGap"
        << std::setw(20)
        << "KE"
        << std::setw(20)
        << "PE"
        << std::setw(20)
        << "Total E"
        << std::setw(20)
        << "E-E0"
        << '\n';

    std::cout
        << std::setw(10)
        << 0
        << std::setw(20)
        << 0.0
        << std::setw(20)
        << minimumGap(sortedR)
        << std::setw(20)
        << KE0
        << std::setw(20)
        << PE0
        << std::setw(20)
        << E0
        << std::setw(20)
        << 0.0
        << '\n';

    for (int step = 1;
         step <= STEPS;
         step++) {

        for (int i = 0; i < N; i++) {
            r[i] +=
                v[i] * DT +
                0.5 *
                a[i] *
                DT *
                DT;
        }

        buildCellList(
            r,
            sortedR,
            sortedIndex,
            cellStart,
            r_min,
            r_max,
            numCells
        );

        cudaFree(d_cellStart);

        cudaMalloc(
            &d_cellStart,
            (numCells + 1) *
            sizeof(int)
        );

        computeForcesGPU(
            sortedR,
            cellStart,
            sortedIndex,
            F,
            r_min,
            numCells,
            d_sortedR,
            d_cellStart,
            d_F_sorted
        );

        for (int i = 0; i < N; i++) {
            a_new[i] =
                F[i] / MASS;
        }

        for (int i = 0; i < N; i++) {
            v[i] +=
                0.5 *
                (a[i] + a_new[i]) *
                DT;
        }

        a.swap(a_new);

        if (step % CHECKPOINT == 0) {

            double KE =
                computeKE(v);

            double PE =
                computePE(
                    r,
                    sortedIndex,
                    cellStart,
                    r_min,
                    numCells
                );

            double E =
                KE + PE;

            double deltaE =
                E - E0;

            double minGap =
                minimumGap(sortedR);

            std::cout
                << std::setw(10)
                << step
                << std::setw(20)
                << step * DT
                << std::setw(20)
                << minGap
                << std::setw(20)
                << KE
                << std::setw(20)
                << PE
                << std::setw(20)
                << E
                << std::setw(20)
                << deltaE
                << '\n';
        }
    }

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [29]:
!nvcc -O2 md_dt1e5.cu -o md_dt1e5
!./md_dt1e5

N = 65536
dt = 1.000000000000e-05
Total simulated time = 1.000000000000e+00
Initial KE = 3.276799999999e+00
Initial PE = -2.460319741960e+04
Initial E  = -2.459992061960e+04

      Step                Time              MinGap                  KE                  PE             Total E                E-E0
         0  0.000000000000e+00  1.100933960746e+00  3.276799999999e+00 -2.460319741960e+04 -2.459992061960e+04  0.000000000000e+00
      5000  5.000000000000e-02  1.104583779203e+00  9.167187015453e+01 -2.469159248970e+04 -2.459992061954e+04  5.377660272643e-08
     10000  1.000000000000e-01  1.116006445578e+00  3.604023839520e+02 -2.496032300344e+04 -2.459992061949e+04  1.112603058573e-07
     15000  1.500000000000e-01  1.126100662615e+00  8.165890252502e+02 -2.541650964476e+04 -2.459992061951e+04  8.863571565598e-08
     20000  2.000000000000e-01  1.118257331545e+00  1.453695281921e+03 -2.605361590165e+04 -2.459992061973e+04 -1.277294359170e-07
     25000  2.500000000000e-01  1.09917

E - E0 at t = 1 is -173.58, against -173.64 before. A 100x change in dt did nothing, so the time step is not the cause. MinGap stays between 0.985 and 1.126 and never collapses, so there is no collision either.

### The cause: the cutoff itself

With a hard cutoff the potential energy is U(r) for r < r_c and 0 beyond, but U(r_c) = -0.0163 is not zero. Every pair that crosses r_c changes the computed energy by 0.0163 while the force switches off with no matching impulse. This depends on the potential model and not on the time step, which fits the dt result. Dividing the drift by the jump, 173.6 / 0.0163, gives about 10,600 net crossings out of 65,535 second-neighbour pairs (initial separation about 3, in a chain that contracts), which is plausible. I did not count the crossings directly.

The fix is the shifted-force potential,

$$U_{sf}(r) = U(r) - U(r_c) - (r - r_c)\,U'(r_c), \qquad F_{sf}(r) = F(r) - F(r_c) \qquad (r < r_c)$$

so that both the potential and the force go to zero at r_c. The constant F(r_c) = -0.039 is computed once and subtracted in the kernel and in the energy sum.

dt = 1e-4, 30,000 steps (t = 3), energy printed every 1000 steps.

In [30]:
%%writefile md_differential.cu
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE_SIZE 256

const double SIGMA = 1.0;
const double EPSILON = 1.0;
const double MASS = 1.0;
const double RC = 2.5;
const double LCELL = 2.5;

double ljForce(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return 24.0 * EPSILON * invR *
           (2.0 * sr12 - sr6);
}

double ljPotential(double r) {
    double invR = 1.0 / r;
    double sr = SIGMA * invR;
    double sr6 = sr * sr * sr * sr * sr * sr;
    double sr12 = sr6 * sr6;

    return 4.0 * EPSILON *
           (sr12 - sr6);
}

const double FC = ljForce(RC);

__global__ void cellListForceKernel(
    const double *sortedR,
    const int *cellStart,
    double *F_sorted,
    int N,
    int numCells,
    double r_min,
    double FC
) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    if (k >= N) {
        return;
    }

    double myR = sortedR[k];

    int myCell =
        (int)floor((myR - r_min) / LCELL);

    if (myCell < 0) {
        myCell = 0;
    }

    if (myCell >= numCells) {
        myCell = numCells - 1;
    }

    double Fi = 0.0;

    for (int c = myCell - 1; c <= myCell + 1; c++) {
        if (c < 0 || c >= numCells) {
            continue;
        }

        for (int m = cellStart[c];
             m < cellStart[c + 1];
             m++) {

            if (m == k) {
                continue;
            }

            double dx = myR - sortedR[m];
            double dist = fabs(dx);

            if (dist < RC) {
                double invR = 1.0 / dist;

                double sr = SIGMA * invR;
                double sr6 =
                    sr * sr * sr * sr * sr * sr;
                double sr12 = sr6 * sr6;

                double force =
                    24.0 * EPSILON * invR *
                    (2.0 * sr12 - sr6) - FC;

                Fi += force * (dx * invR);
            }
        }
    }

    F_sorted[k] = Fi;
}

void buildCellList(
    const std::vector<double>& r,
    std::vector<double>& sortedR,
    std::vector<int>& sortedIndex,
    std::vector<int>& cellStart,
    double& r_min,
    double& r_max,
    int& numCells
) {
    int N = r.size();

    r_min = r[0];
    r_max = r[0];

    for (int i = 1; i < N; i++) {
        r_min = std::min(r_min, r[i]);
        r_max = std::max(r_max, r[i]);
    }

    numCells =
        (int)ceil((r_max - r_min) / LCELL);

    if (numCells < 1) {
        numCells = 1;
    }

    std::vector<int> cellCount(numCells, 0);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        cellCount[cell]++;
    }

    cellStart.assign(numCells + 1, 0);

    for (int c = 0; c < numCells; c++) {
        cellStart[c + 1] =
            cellStart[c] + cellCount[c];
    }

    std::vector<int> cursor = cellStart;

    sortedR.resize(N);
    sortedIndex.resize(N);

    for (int i = 0; i < N; i++) {
        int cell =
            (int)floor((r[i] - r_min) / LCELL);

        if (cell < 0) {
            cell = 0;
        }

        if (cell >= numCells) {
            cell = numCells - 1;
        }

        int slot = cursor[cell];

        sortedR[slot] = r[i];
        sortedIndex[slot] = i;

        cursor[cell]++;
    }
}

void computeForcesGPU(
    const std::vector<double>& sortedR,
    const std::vector<int>& cellStart,
    const std::vector<int>& sortedIndex,
    std::vector<double>& F,
    double r_min,
    int numCells,
    double *d_sortedR,
    int *d_cellStart,
    double *d_F_sorted,
    double FC
) {
    int N = sortedR.size();

    cudaMemcpy(
        d_sortedR,
        sortedR.data(),
        N * sizeof(double),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_cellStart,
        cellStart.data(),
        (numCells + 1) * sizeof(int),
        cudaMemcpyHostToDevice
    );

    int blocks =
        (N + TILE_SIZE - 1) / TILE_SIZE;

    cellListForceKernel<<<blocks, TILE_SIZE>>>(
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        N,
        numCells,
        r_min,
        FC
    );

    cudaError_t error = cudaGetLastError();

    if (error != cudaSuccess) {
        std::cerr
            << "Kernel error: "
            << cudaGetErrorString(error)
            << '\n';
        exit(1);
    }

    std::vector<double> F_sorted(N);

    cudaMemcpy(
        F_sorted.data(),
        d_F_sorted,
        N * sizeof(double),
        cudaMemcpyDeviceToHost
    );

    for (int k = 0; k < N; k++) {
        F[sortedIndex[k]] = F_sorted[k];
    }
}

double computeKE(const std::vector<double>& v) {
    double KE = 0.0;

    for (double vi : v) {
        KE += 0.5 * MASS * vi * vi;
    }

    return KE;
}

double computePE(
    const std::vector<double>& r,
    const std::vector<int>& sortedIndex,
    const std::vector<int>& cellStart,
    double r_min,
    int numCells,
    double FC
) {
    int N = r.size();
    double PE = 0.0;

    for (int i = 0; i < N; i++) {
        int myCell =
            (int)floor((r[i] - r_min) / LCELL);

        if (myCell < 0) {
            myCell = 0;
        }

        if (myCell >= numCells) {
            myCell = numCells - 1;
        }

        for (int c = myCell - 1; c <= myCell + 1; c++) {
            if (c < 0 || c >= numCells) {
                continue;
            }

            for (int m = cellStart[c];
                 m < cellStart[c + 1];
                 m++) {

                int j = sortedIndex[m];

                if (j <= i) {
                    continue;
                }

                double dx = r[i] - r[j];
                double dist = fabs(dx);

                if (dist < RC) {
                    PE += ljPotential(dist) - ljPotential(RC)
                        + FC * (dist - RC);
                }
            }
        }
    }

    return PE;
}

int main() {

    const int N = 65536;
    const int STEPS = 30000;
    const double DT = 0.0001;

    const bool RANDOM_VELOCITIES = false;

    std::vector<double> r(N);
    std::vector<double> v(N);
    std::vector<double> a(N);
    std::vector<double> a_new(N);
    std::vector<double> F(N);

    double spacing = 1.5;
    double jitterAmplitude = 0.2;

    std::mt19937 gen(42);

    std::uniform_real_distribution<double> jitter(
        -jitterAmplitude,
        jitterAmplitude
    );

    for (int i = 0; i < N; i++) {
        r[i] =
            i * spacing +
            jitter(gen);
    }

    if (RANDOM_VELOCITIES) {

        std::uniform_real_distribution<double> velocityDistribution(
            -0.01,
            0.01
        );

        double meanVelocity = 0.0;

        for (int i = 0; i < N; i++) {
            v[i] = velocityDistribution(gen);
            meanVelocity += v[i];
        }

        meanVelocity /= N;

        for (int i = 0; i < N; i++) {
            v[i] -= meanVelocity;
        }

    } else {

        for (int i = 0; i < N; i++) {
            if (i % 2 == 0) {
                v[i] = 0.01;
            } else {
                v[i] = -0.01;
            }
        }
    }

    std::vector<double> sortedR;
    std::vector<int> sortedIndex;
    std::vector<int> cellStart;

    double r_min;
    double r_max;
    int numCells;

    buildCellList(
        r,
        sortedR,
        sortedIndex,
        cellStart,
        r_min,
        r_max,
        numCells
    );

    double *d_sortedR;
    int *d_cellStart;
    double *d_F_sorted;

    cudaMalloc(
        &d_sortedR,
        N * sizeof(double)
    );

    cudaMalloc(
        &d_cellStart,
        (numCells + 1) * sizeof(int)
    );

    cudaMalloc(
        &d_F_sorted,
        N * sizeof(double)
    );

    computeForcesGPU(
        sortedR,
        cellStart,
        sortedIndex,
        F,
        r_min,
        numCells,
        d_sortedR,
        d_cellStart,
        d_F_sorted,
        FC
    );

    for (int i = 0; i < N; i++) {
        a[i] = F[i] / MASS;
    }

    double KE0 =
        computeKE(v);

    double PE0 =
        computePE(
            r,
            sortedIndex,
            cellStart,
            r_min,
            numCells,
            FC
        );

    double E0 = KE0 + PE0;

    std::cout
        << std::scientific
        << std::setprecision(12);

    std::cout << "N = " << N << '\n';
    std::cout << "dt = " << DT << '\n';
    std::cout << "Random velocities = "
              << (RANDOM_VELOCITIES ? "yes" : "no")
              << '\n';
    std::cout << "Initial KE = " << KE0 << '\n';
    std::cout << "Initial PE = " << PE0 << '\n';
    std::cout << "Initial E  = " << E0 << '\n';

    std::cout << "\n";

    std::cout
        << std::setw(8) << "Step"
        << std::setw(20) << "KE"
        << std::setw(20) << "PE"
        << std::setw(20) << "Total E"
        << std::setw(20) << "E-E0"
        << '\n';

    std::cout
        << std::setw(8) << 0
        << std::setw(20) << KE0
        << std::setw(20) << PE0
        << std::setw(20) << E0
        << std::setw(20) << 0.0
        << '\n';

    for (int step = 1; step <= STEPS; step++) {

        for (int i = 0; i < N; i++) {
            r[i] +=
                v[i] * DT +
                0.5 * a[i] * DT * DT;
        }

        buildCellList(
            r,
            sortedR,
            sortedIndex,
            cellStart,
            r_min,
            r_max,
            numCells
        );

        cudaFree(d_cellStart);

        cudaMalloc(
            &d_cellStart,
            (numCells + 1) * sizeof(int)
        );

        computeForcesGPU(
            sortedR,
            cellStart,
            sortedIndex,
            F,
            r_min,
            numCells,
            d_sortedR,
            d_cellStart,
            d_F_sorted,
            FC
        );

        for (int i = 0; i < N; i++) {
            a_new[i] =
                F[i] / MASS;
        }

        for (int i = 0; i < N; i++) {
            v[i] +=
                0.5 *
                (a[i] + a_new[i]) *
                DT;
        }

        if (step % 1000 == 0) {

            double KE =
                computeKE(v);

            double PE =
                computePE(
                    r,
                    sortedIndex,
                    cellStart,
                    r_min,
                    numCells,
                    FC
                );

            double E = KE + PE;
            double deltaE = E - E0;

            std::cout
                << std::setw(8)
                << step
                << std::setw(20)
                << KE
                << std::setw(20)
                << PE
                << std::setw(20)
                << E
                << std::setw(20)
                << deltaE
                << '\n';
        }

        a.swap(a_new);
    }

    cudaFree(d_sortedR);
    cudaFree(d_cellStart);
    cudaFree(d_F_sorted);

    return 0;
}

In [31]:
!nvcc -O2 md_differential.cu -o md_differential
!./md_differential

N = 65536
dt = 1.000000000000e-04
Random velocities = no
Initial KE = 3.276799999999e+00
Initial PE = -2.097802760502e+04
Initial E  = -2.097475080502e+04

    Step                  KE                  PE             Total E                E-E0
       0  3.276799999999e+00 -2.097802760502e+04 -2.097475080502e+04  0.000000000000e+00
    1000  3.604009330176e+02 -2.133515173512e+04 -2.097475080210e+04  2.916960511357e-06
    2000  1.453690085208e+03 -2.242844090698e+04 -2.097475082177e+04 -1.675026578596e-05
    3000  2.956640342607e+03 -2.393139129069e+04 -2.097475094808e+04 -1.430654956494e-04
    4000  3.914654791410e+03 -2.488940577601e+04 -2.097475098460e+04 -1.795760035748e-04
    5000  5.178995462381e+03 -2.615374658365e+04 -2.097475112127e+04 -3.162499924656e-04
    6000  6.687444257468e+03 -2.766219556514e+04 -2.097475130767e+04 -5.026502476539e-04
    7000  7.075629782149e+03 -2.805038109947e+04 -2.097475131732e+04 -5.122985021444e-04
    8000  7.319758478384e+03 -2.82945098526

At t = 1 (step 10,000), E - E0 is -6.5e-4 instead of -173.6, about 270,000 times smaller. The first 10,000 steps show a slow monotonic descent; after that the error stays between about -6.2e-4 and -7.2e-4 up to t = 3. Relative to the total energy of -2.0975e4 that is about 3e-8.

The kinetic energy rises from 3.3 to about 9.5e3 while the potential energy falls by the same amount. The starting spacing of 1.5 is wider than the equilibrium spacing of 1.12, so the chain contracts and converts potential energy into kinetic energy; the total stays constant, which is what matters here.

The remaining 3e-8 is checked against OpenMM in part 2 (at N = 4,096). OpenMM reproduces it, so it is a property of the model and time step and not a bug in this code.

## 7. Summary

| Step | Result at N = 65,536 |
|---|---|
| Naive kernel | 1514 ms, 45.4 GFLOP/s, 17.8% of FP64 peak |
| One division per pair | 1.71x faster (1514 to 887 ms); a spot-check exposed unusable test data |
| Lattice-plus-jitter positions | spot-check error 4.4e-16, same speedup |
| Shared-memory tiling | about 1% faster, as predicted |
| Cell lists | 0.1785 ms, about 5,000x faster, uses a cutoff |
| 30,000-step dynamics | energy error about 3e-8 of the total after the cutoff fix |

Predictions were written down before each measurement, and several were wrong: the energy drift was blamed first on the time step, then on the initial velocities, then on a collision, before the cutoff was identified.

### Limits

- One dimension only, with about two interacting neighbours per particle.
- One GPU, double precision only in this part.
- Kernel timings are the minimum and mean of 20 runs in a single session.
- No profiler data, so explanations of what limits a kernel are inferences from timings.
- The cell list is rebuilt on the host in the dynamics loop. Part 2 measures what that costs.